In [7]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

# Prostate158

Complete

In [21]:
DATASET_NAME = "Prostate158"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



15:35:59 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:82


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver'
]

In [9]:
description = """
Prostate158 is a curated dataset of 3 biparametric 3 Tesla prostate MRI images for automatic segmentation of anatomical zones and carcinomatous lesions. Histopathologic confirmation is available for each cancerous lesion.

All studies include a T2-weighted (T2W) and diffusion-weighted (DWI) images with apparent diffusion coefficient (ADC) maps. Images in each study were resampled so that orientation, direction, and spacing were the same.
"""

In [10]:
source = ZenodoSource(
    file_type="nifti",
    record_id="6481141",
    description=description,
    post_download=["unzip"]
)

In [15]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
print(dataset_index_path)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

indexed_datasets/Prostate158

In [12]:

print(indexed_datasets.source_config(DATASET_NAME))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='zenodo',
    record_id='6481141',
    filenames=None,
    post_download=['unzip'],
    description='\nProstate158 is a curated dataset of 3 biparametric 3 Tesla prostate MRI images for automatic 
segmentation of anatomical zones and carcinomatous lesions. Histopathologic confirmation is available for each 
cancerous lesion.\n\nAll studies include a T2-weighted (T2W) and diffusion-weighted (DWI) images with apparent 
diffusion coefficient (ADC) maps. Images in each study were resampled so that orientation, direction, and spacing 
were the same.\n'
)

In [17]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
print(temp_data_path.absolute())
#downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

/home/gpudual/bhklab/declan/med-image_index/temp_data/Prostate158

In [23]:
import zipfile
from glob import glob
import os

def extract_zip(zip_path, extract_to=None):
    """
    Extract a zip file to a specified directory
    
    Args:
        zip_path: Path to the zip file
        extract_to: Directory to extract to (default: same directory as zip)
    """
    if extract_to is None:
        extract_to = os.path.dirname(zip_path)
    
    # Create extraction directory if it doesn't exist
    os.makedirs(extract_to, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"Extracted {zip_path} to {extract_to}")

for zip_file in glob(f"{temp_data_path}/*.zip"):
    print(zip_file)
    # Get the base name without extension to create a folder with the same name
    zip_name = os.path.splitext(os.path.basename(zip_file))[0]
    extract_to = os.path.join(f"{temp_data_path}", zip_name)
    extract_zip(zip_file, temp_data_path)

temp_data/Prostate158/prostate158_train.zip

Extracted temp_data/Prostate158/prostate158_train.zip to temp_data/Prostate158

In [27]:
crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="prostate158_train/train/{reference_id}/{phase}.nii.gz",
    mask_name_pattern="prostate158_train/train/{reference_id}/{phase}_{mask_type}_reader{scan_number:d}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=True,
)
crawler.crawl()


Parsing 844 files:   0%|                               | 0/844 [00:00<?, ?it/s]11:43:11 [warning  ] File prostate158_train/train/020/adc_tumor_reader1.nii.gz matched both scan and mask patterns. Returning mask match. [imgtools] call=parse_niftis._match_file:126
11:43:11 [warning  ] File prostate158_train/train/020/adc_tumor_reader2.nii.gz matched both scan and mask patterns. Returning mask match. [imgtools] call=parse_niftis._match_file:126
11:43:11 [warning  ] File prostate158_train/train/020/t2_anatomy_reader1.nii.gz matched both scan and mask patterns. Returning mask match. [imgtools] call=parse_niftis._match_file:126
11:43:11 [warning  ] File prostate158_train/train/020/t2_tumor_reader1.nii.gz matched both scan and mask patterns. Returning mask match. [imgtools] call=parse_niftis._match_file:126
11:43:11 [warning  ] File prostate158_train/train/021/adc_tumor_reader1.nii.gz matched both scan and mask patterns. Returning mask match. [imgtools] call=parse_niftis._match_file:126
11:43:

In [28]:
index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

index_df



,reference_id,phase,mask_type,scan_number,filepath,file_type,class,hash,size,ndim,...,mask.bbox.max_coord,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan
0,020_adc,adc,NaN,NaN,prostate158_train/train/020/adc.nii.gz,scan,MedImage,4e3282075ce042149a70dacc3df170a631741566,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,020_adc,adc,tumor,1.0,prostate158_train/train/020/adc_tumor_reader1....,mask,Mask,f404fdd6648fffecfe514e86b57a8ce0702901c5,"(270, 270, 24)",3,...,"(161, 161, 10)",15.492265,0.819866,1.452138,1.355063,5.660848,402.691899,"(7.9784719968769595, 11.58584149199298, 15.699...",1.0,prostate158_train/train/020/adc.nii.gz
2,020_adc,adc,tumor,2.0,prostate158_train/train/020/adc_tumor_reader2....,mask,Mask,cd537fbf65d90a3d3f4db313e89b7716249c2fed,"(270, 270, 24)",3,...,"(154, 158, 9)",12.058600,0.826445,1.575372,1.077764,4.331706,235.791320,"(6.241068200622396, 9.832005154075304, 10.5965...",1.0,prostate158_train/train/020/adc.nii.gz
3,020_dwi,dwi,NaN,NaN,prostate158_train/train/020/dwi.nii.gz,scan,MedImage,19f67b3b09de122cf9f62f8ea71e4f018e00b907,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,020_t2,t2,NaN,NaN,prostate158_train/train/020/t2.nii.gz,scan,MedImage,d2b3fe9bc8584a854f8ff27ad0c101da2dabc6a0,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,158_adc,adc,tumor,2.0,prostate158_train/train/158/adc_tumor_reader2....,mask,Mask,dcf5bdc5753a1854f2da9c07e46a2aff63ea1b26,"(442, 442, 25)",3,...,"(291, 220, 14)",22.047915,0.789130,1.615034,1.157103,7.850040,774.378972,"(10.86400671286656, 17.545743648554094, 20.302...",1.0,prostate158_train/train/158/adc.nii.gz
840,158_dwi,dwi,NaN,NaN,prostate158_train/train/158/dwi.nii.gz,scan,MedImage,7b14628f825d8bbd893f669423c367a263268558,"(442, 442, 25)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
841,158_t2,t2,NaN,NaN,prostate158_train/train/158/t2.nii.gz,scan,MedImage,a6ca6810dbda0bbd34fa64ab070b55b6b25347fc,"(442, 442, 25)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
842,158_t2,t2,anatomy,1.0,prostate158_train/train/158/t2_anatomy_reader1...,mask,Mask,15bb9ecbca4ef112d3471266b69d627fee9fb532,"(442, 442, 25)",3,...,"(336, 347, 21)",60.898906,0.809331,1.227566,1.127483,24.614715,7613.765166,"(41.25643268446543, 50.64498484729952, 57.1013...",1.0,prostate158_train/train/158/t2.nii.gz


In [29]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "MR"
    else:
        return "SEG"
    
def map_roi(row):
    if row["file_type"] == "scan":
            return ""
    else:
        if row["mask_type"] == "tumor":
            return "TUMOR"
        elif row["mask_type"] == "anatomy":
            return "PROSTATE"
    
index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["ROINames"] = index_df.apply(map_modality, axis=1)

index_df["BodyPartExamined"] = "PROSTATE"




In [30]:
index_df

,reference_id,phase,mask_type,scan_number,filepath,file_type,class,hash,size,ndim,...,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,ROINames,BodyPartExamined
0,020_adc,adc,NaN,NaN,prostate158_train/train/020/adc.nii.gz,scan,MedImage,4e3282075ce042149a70dacc3df170a631741566,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
1,020_adc,adc,tumor,1.0,prostate158_train/train/020/adc_tumor_reader1....,mask,Mask,f404fdd6648fffecfe514e86b57a8ce0702901c5,"(270, 270, 24)",3,...,1.452138,1.355063,5.660848,402.691899,"(7.9784719968769595, 11.58584149199298, 15.699...",1.0,prostate158_train/train/020/adc.nii.gz,SEG,SEG,PROSTATE
2,020_adc,adc,tumor,2.0,prostate158_train/train/020/adc_tumor_reader2....,mask,Mask,cd537fbf65d90a3d3f4db313e89b7716249c2fed,"(270, 270, 24)",3,...,1.575372,1.077764,4.331706,235.791320,"(6.241068200622396, 9.832005154075304, 10.5965...",1.0,prostate158_train/train/020/adc.nii.gz,SEG,SEG,PROSTATE
3,020_dwi,dwi,NaN,NaN,prostate158_train/train/020/dwi.nii.gz,scan,MedImage,19f67b3b09de122cf9f62f8ea71e4f018e00b907,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
4,020_t2,t2,NaN,NaN,prostate158_train/train/020/t2.nii.gz,scan,MedImage,d2b3fe9bc8584a854f8ff27ad0c101da2dabc6a0,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,158_adc,adc,tumor,2.0,prostate158_train/train/158/adc_tumor_reader2....,mask,Mask,dcf5bdc5753a1854f2da9c07e46a2aff63ea1b26,"(442, 442, 25)",3,...,1.615034,1.157103,7.850040,774.378972,"(10.86400671286656, 17.545743648554094, 20.302...",1.0,prostate158_train/train/158/adc.nii.gz,SEG,SEG,PROSTATE
840,158_dwi,dwi,NaN,NaN,prostate158_train/train/158/dwi.nii.gz,scan,MedImage,7b14628f825d8bbd893f669423c367a263268558,"(442, 442, 25)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
841,158_t2,t2,NaN,NaN,prostate158_train/train/158/t2.nii.gz,scan,MedImage,a6ca6810dbda0bbd34fa64ab070b55b6b25347fc,"(442, 442, 25)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
842,158_t2,t2,anatomy,1.0,prostate158_train/train/158/t2_anatomy_reader1...,mask,Mask,15bb9ecbca4ef112d3471266b69d627fee9fb532,"(442, 442, 25)",3,...,1.227566,1.127483,24.614715,7613.765166,"(41.25643268446543, 50.64498484729952, 57.1013...",1.0,prostate158_train/train/158/t2.nii.gz,SEG,SEG,PROSTATE


In [31]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet

index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

,SampleID,phase,mask_type,scan_number,filepath,file_type,class,hash,size,ndim,...,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,Modality,ROINames,BodyPartExamined
0,020_adc,adc,NaN,NaN,prostate158_train/train/020/adc.nii.gz,scan,MedImage,4e3282075ce042149a70dacc3df170a631741566,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
1,020_adc,adc,tumor,1.0,prostate158_train/train/020/adc_tumor_reader1....,mask,Mask,f404fdd6648fffecfe514e86b57a8ce0702901c5,"(270, 270, 24)",3,...,1.452138,1.355063,5.660848,402.691899,"(7.9784719968769595, 11.58584149199298, 15.699...",1.0,prostate158_train/train/020/adc.nii.gz,SEG,SEG,PROSTATE
2,020_adc,adc,tumor,2.0,prostate158_train/train/020/adc_tumor_reader2....,mask,Mask,cd537fbf65d90a3d3f4db313e89b7716249c2fed,"(270, 270, 24)",3,...,1.575372,1.077764,4.331706,235.791320,"(6.241068200622396, 9.832005154075304, 10.5965...",1.0,prostate158_train/train/020/adc.nii.gz,SEG,SEG,PROSTATE
3,020_dwi,dwi,NaN,NaN,prostate158_train/train/020/dwi.nii.gz,scan,MedImage,19f67b3b09de122cf9f62f8ea71e4f018e00b907,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
4,020_t2,t2,NaN,NaN,prostate158_train/train/020/t2.nii.gz,scan,MedImage,d2b3fe9bc8584a854f8ff27ad0c101da2dabc6a0,"(270, 270, 24)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,158_adc,adc,tumor,2.0,prostate158_train/train/158/adc_tumor_reader2....,mask,Mask,dcf5bdc5753a1854f2da9c07e46a2aff63ea1b26,"(442, 442, 25)",3,...,1.615034,1.157103,7.850040,774.378972,"(10.86400671286656, 17.545743648554094, 20.302...",1.0,prostate158_train/train/158/adc.nii.gz,SEG,SEG,PROSTATE
840,158_dwi,dwi,NaN,NaN,prostate158_train/train/158/dwi.nii.gz,scan,MedImage,7b14628f825d8bbd893f669423c367a263268558,"(442, 442, 25)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
841,158_t2,t2,NaN,NaN,prostate158_train/train/158/t2.nii.gz,scan,MedImage,a6ca6810dbda0bbd34fa64ab070b55b6b25347fc,"(442, 442, 25)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MR,MR,PROSTATE
842,158_t2,t2,anatomy,1.0,prostate158_train/train/158/t2_anatomy_reader1...,mask,Mask,15bb9ecbca4ef112d3471266b69d627fee9fb532,"(442, 442, 25)",3,...,1.227566,1.127483,24.614715,7613.765166,"(41.25643268446543, 50.64498484729952, 57.1013...",1.0,prostate158_train/train/158/t2.nii.gz,SEG,SEG,PROSTATE
